In [2]:
from pymed import PubMed
import os
import json
import time
import regex as re
import xmltodict, json
# ElementTree
import xml.etree.ElementTree as ET

In [12]:
# Create a PubMed object that GraphQL can use to query
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyRetrivalTool", email="c.g.a.viviers@tue.nl")

# Create a GraphQL query in plain text

# Example
# query = '(("2018/05/01"[Date - Create] : "3000"[Date - Create])) AND (Xiaoying Xian[Author] OR diabetes)'
# Get all the articles that contain the word "invivo" in the title and were publish in the last 5 years
query = '(("2016/05/01"[Date - Create] : "2024/05/01"[Date - Create])) AND in-vivo[Title]'


# Execute the query against the API
results = pubmed.query(query, max_results=50)

In [13]:
results

In [14]:
failed_articles = []

# Write each article to a file

# Create a directory to store the articles
if not os.path.exists('articles'):
    os.makedirs('articles')

# Loop over the retrieved articles
for article in results:

    # Extract and format all the information from the article
    article_dict = article.toDict()

    # make sure all the data is json serializable, 
    for key in article_dict.keys():
        if not isinstance(article_dict[key], (str, int, float, list, dict)):
           # if xml object, convert to string
            if isinstance(article_dict[key], ET.Element):
                article_dict[key] = ET.tostring(article_dict[key], encoding='utf8').decode('utf8')
            # if xml string, convert to dict
            elif isinstance(article_dict[key], str) and article_dict[key].startswith('<'):
                article_dict[key] = xmltodict.parse(article_dict[key])
            else:
           
             article_dict[key] = str(article_dict[key])

    
    # Create a file name for the article from the paper title, use regex to remove unwanted characters and ensure a valid file name
    file_name = re.sub(r'[^\w\s-]', '', article_dict['title'])
    file_name = re.sub(r'[-\s]+', '-', file_name)
    file_name = f'articles/{file_name}.json'
    

    # Write the article to a file
    try:
        with open(file_name, 'w') as file:
            json.dump(article_dict, file)
    except Exception as e:
        print(f'Error writing file {file_name}')
        print(e)
        failed_articles.append(file_name)


In [15]:
# loop through failed articles check if all have abstracts
local_results =  os.listdir('articles')


for article in local_results:
    article_path = f'articles/{article}'
    with open(article_path, 'r') as file:
        data = json.load(file)
        if 'abstract' not in data.keys():
            print(f"Article {article} does not have an abstract")
            # failed_articles.append(article)

    